In [8]:
from sysdata.config.configdata import Config
my_config=Config()
my_config

Config with elements: 

In [14]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData
from systems.basesystem import System

from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac
from systems.forecasting import Rules
from systems.trading_rules import TradingRule

data=csvFuturesSimData()

ewmac_8=TradingRule((ewmac, [], dict(Lfast=8, Lslow=32))) ## as a tuple (function, data, other_args) notice the empty element in the middle
ewmac_32=TradingRule(dict(function=ewmac, other_args=dict(Lfast=32, Lslow=128)))  ## as a dict
my_rules=Rules(dict(ewmac8=ewmac_8, ewmac32=ewmac_32))
my_rules.trading_rules()['ewmac32']

empty_rules=Rules()
my_config.trading_rules=dict(ewmac8=ewmac_8, ewmac32=ewmac_32)
my_system=System([empty_rules], data, my_config)
print(my_system.rules.get_raw_forecast("SOFR", "ewmac8"))

2025-03-31 19:27:27 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:27:27 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:27:27 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:27:27 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:27:27 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac8
index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.341102
2024-03-25   -0.319042
2024-03-26   -0.283304
2024-03-27   -0.209219
2024-03-28   -0.168111
Freq: B, Name: price, Length: 10440, dtype: float64


In [16]:
from systems.forecast_scale_cap import ForecastScaleCap


## By default we pool estimates across instruments. It's worth telling the system what instruments we want to use:
#
my_config.instruments=["SOFR", "US10", "CORN", "SP500_micro"]

## this parameter ensures we estimate:
my_config.use_forecast_scale_estimates=True

fcs=ForecastScaleCap()
my_system = System([fcs, my_rules], data, my_config)
print(my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5))

2025-03-31 19:28:20 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:28:20 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:28:20 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:28:20 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:28:20 DEBUG base_system {'stage': 'forecastScaleCap'} Getting cross sectional forecasts for scalar calculation for ewmac32 over CORN, SOFR, SP500_micro, US10
2025-03-31 19:28:20 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2025-03-31 19:28:20 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2025-03-31 19:28:20 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500_micro'} Calculating raw forecast

In [17]:
my_config.forecast_scalars=dict(ewmac8=5.3, ewmac32=2.65)

## this parameter ensures we don't estimate:
my_config.use_forecast_scale_estimates=False

my_system=System([fcs, empty_rules], data, my_config)

print(my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32"))
2.65 # now a single float value

my_system.forecastScaleCap.get_capped_forecast("SOFR", "ewmac32")

2025-03-31 19:28:34 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:28:34 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:28:34 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:28:34 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:28:34 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
index
1984-03-23    2.65
1984-03-26    2.65
1984-03-27    2.65
1984-03-28    2.65
1984-03-29    2.65
              ... 
2024-03-22    2.65
2024-03-25    2.65
2024-03-26    2.65
2024-03-27    2.65
2024-03-28    2.65
Freq: B, Length: 10440, dtype: float64
2025-03-31 19:28:34 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32


index
1984-03-23         NaN
1984-03-26         NaN
1984-03-27         NaN
1984-03-28         NaN
1984-03-29         NaN
                ...   
2024-03-22   -0.398947
2024-03-25   -0.434692
2024-03-26   -0.456727
2024-03-27   -0.443496
2024-03-28   -0.442636
Freq: B, Length: 10440, dtype: float64

In [21]:
from systems.forecast_combine import ForecastCombine

combiner = ForecastCombine()
my_system = System([fcs, empty_rules, combiner], data, my_config)

print(my_system.combForecast.get_forecast_diversification_multiplier("SOFR").tail(5))
my_system.combForecast.get_forecast_weights("SOFR").tail(5)

2025-03-31 19:30:07 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:30:07 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:30:07 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:30:07 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:30:07 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Using fixed FDM multiplier of 1.000 for SOFR
2025-03-31 19:30:07 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32
2025-03-31 19:30:07 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2025-03-31 19:30:07 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


,ewmac8,ewmac32
index,,
2024-03-22,0.5,0.5
2024-03-25,0.5,0.5
2024-03-26,0.5,0.5
2024-03-27,0.5,0.5
2024-03-28,0.5,0.5


In [22]:
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing
from systems.accounts.accounts_stage import Account
combiner = ForecastCombine()
raw_data = RawData()
position_size = PositionSizing()
my_account = Account()

## let's use naive markowitz to get more interesting results...
my_config.forecast_weight_estimate = dict(method="one_period")
my_config.use_forecast_weight_estimates = True
my_config.use_forecast_div_mult_estimates = True

combiner = ForecastCombine()
my_system = System([my_account, fcs, my_rules, combiner, position_size, raw_data], data, my_config)

print(my_system.combForecast.get_forecast_weights("US10").tail(5))
print(my_system.combForecast.get_forecast_diversification_multiplier("US10").tail(5))


2025-03-31 19:30:28 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:30:28 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:30:28 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:30:28 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:30:28 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating forecast weights for US10
2025-03-31 19:30:28 INFO base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating raw forecast weights for US10
2025-03-31 19:30:28 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac32
2025-03-31 19:30:28 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2025-03-31 1

In [23]:
my_config.forecast_weights=dict(ewmac8=0.5, ewmac32=0.5)
my_config.forecast_div_multiplier=1.1
my_config.use_forecast_weight_estimates = False
my_config.use_forecast_div_mult_estimates = False
my_system=System([fcs, empty_rules, combiner, raw_data, position_size], data, my_config)
my_system.combForecast.get_combined_forecast("SOFR").tail(5)

2025-03-31 19:30:57 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:30:57 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:30:57 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:30:57 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:30:57 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating combined forecast for SOFR
2025-03-31 19:30:57 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32
2025-03-31 19:30:57 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2025-03-31 19:30:57 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac8
2025

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


index
2024-03-22   -1.213732
2024-03-25   -1.169088
2024-03-26   -1.077031
2024-03-27   -0.853796
2024-03-28   -0.733494
Freq: B, dtype: float64

In [24]:
my_config.percentage_vol_target=25
my_config.notional_trading_capital=500000
my_config.base_currency="GBP"

my_system=System([ fcs, empty_rules, combiner, position_size, raw_data], data, my_config)

my_system.positionSize.get_subsystem_position("SOFR").tail(5)

2025-03-31 19:31:15 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:31:15 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:31:15 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:31:15 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:31:15 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating subsystem position for SOFR
2025-03-31 19:31:15 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating volatility scalar for SOFR
2025-03-31 19:31:15 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating instrument value vol for SOFR
2025-03-31 19:31:15 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating instrument currency vol for SOFR
2025-03-3

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


index
2024-03-22   -7.674695
2024-03-25   -7.375864
2024-03-26   -6.913289
2024-03-27   -5.561603
2024-03-28   -4.856693
Freq: B, dtype: float64

In [27]:
from systems.portfolio import Portfolios
portfolio = Portfolios()

## Using shrinkage will speed things but - but I don't recommend it for actual trading...
my_config.use_instrument_weight_estimates = True
my_config.use_instrument_div_mult_estimates = True
my_config.instrument_weight_estimate=dict(method="shrinkage", date_method="in_sample") ## speeds things up

my_system = System([my_account, fcs, my_rules, combiner, position_size, raw_data,
                    portfolio], data, my_config)

print(my_system.portfolio.get_instrument_weights())
print(my_system.portfolio.get_instrument_diversification_multiplier())

2025-03-31 19:33:53 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:33:53 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:33:53 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:33:53 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:33:53 INFO base_system {'stage': 'portfolio'} Calculating instrument weights
2025-03-31 19:33:53 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:33:53 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:33:53 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']
2025-03-31 19:33:53 DEBUG base_system Following instruments have restricted trading:  ['RESTRICTED_EXAMP

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:33:54 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily prices for SOFR
2025-03-31 19:33:54 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SOFR
2025-03-31 19:33:54 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Non-nested dict of forecast weights for SOFR {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:33:54 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Using fixed FDM multiplier of 1.100 for SOFR
2025-03-31 19:33:54 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} No mapping applied for SOFR
2025-03-31 19:33:54 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500_micro'} Calculating subsystem position for SP500_micro
2025-03-31 19:33:54 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500_micro'} Calcul

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:33:54 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily prices for SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Non-nested dict of forecast weights for SP500_micro {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Using fixed FDM multiplier of 1.100 for SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} No mapping applied for SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating subsystem position for US10
2025-03-31 19:33:55 DEBUG base_system {'stage': 

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:33:55 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for US10
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Non-nested dict of forecast weights for US10 {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Using fixed FDM multiplier of 1.100 for US10
2025-03-31 19:33:55 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} No mapping applied for US10
2025-03-31 19:33:55 DEBUG base_system {'stage': 'portfolio'} Following instruments will have zero weight in optimisation of instrument weights as they have no positions (possibly too expensive?) []
2025-03-31 19:33:55 DEBUG base_system {'stage': 'ac

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:33:55 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SOFR'} Calculating pandl for subsystem for instrument SOFR
2025-03-31 19:33:55 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating buffers for SOFR
2025-03-31 19:33:55 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SOFR'} Calculating forecast method buffers for SOFR
2025-03-31 19:33:55 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SOFR'} Calculating buffered subsystem positions
2025-03-31 19:33:55 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily prices for SOFR
2025-03-31 19:33:55 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SP500_micro'} Calculating pandl for subsystem for instrument SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500_micro'} Calculating buffers for SP500_micro
2025-03-31 19:33:55 DEBUG base_system {'stage': 'positionSize', 'instr

In [28]:
my_config.instrument_weights=dict(US10=.1, SOFR=.4, CORN=.3, SP500_micro=.8)
my_config.instrument_div_multiplier=1.5
my_config.use_instrument_weight_estimates = False
my_config.use_instrument_div_mult_estimates = False

my_system=System([ fcs, empty_rules, combiner, position_size, raw_data, portfolio], data, my_config)

my_system.portfolio.get_notional_position("SOFR").tail(5)

2025-03-31 19:34:29 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:34:29 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:34:29 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:34:29 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:34:29 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating notional position for SOFR
2025-03-31 19:34:29 INFO base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating instrument weights
2025-03-31 19:34:29 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating raw instrument weights
2025-03-31 19:34:29 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:34:29 DEBUG base_system Following instrum

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast weights for SOFR
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} You need an accounts stage in the system to estimate forecast costs for SOFR ewmac32. Using costs of zero
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} You need an accounts stage in the system to estimate forecast costs for SOFR ewmac8. Using costs of zero
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SOFR
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Non-nested dict of forecast weights for SOFR {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Using f

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:30 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500_micro'} Calculating capped forecast for SP500_micro ewmac8
2025-03-31 19:34:30 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500_micro'} Calculating raw forecast SP500_micro for ewmac8
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Calculating forecast weights for SP500_micro
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} You need an accounts stage in the system to estimate forecast costs for SP500_micro ewmac32. Using costs of zero
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} You need an accounts stage in the system to estimate forecast costs for SP500_micro ewmac8. Using costs of zero
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Only this set of rules ['ewmac32', 'ewmac8

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:30 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'US10'} Calculating capped forecast for US10 ewmac8
2025-03-31 19:34:30 DEBUG base_system {'stage': 'rules', 'instrument_code': 'US10'} Calculating raw forecast US10 for ewmac8
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating forecast weights for US10
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'US10'} You need an accounts stage in the system to estimate forecast costs for US10 ewmac32. Using costs of zero
2025-03-31 19:34:30 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'US10'} You need an accounts stage in the system to estimate forecast costs for US10 ewmac8. Using costs of zero
2025-03-31 19:34:30 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for US10
2025-03-31 19:34:30 DEBUG base_system {'

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


index
2024-03-22   -2.878011
2024-03-25   -2.765949
2024-03-26   -2.592483
2024-03-27   -2.085601
2024-03-28   -1.821260
Freq: B, dtype: float64

In [29]:
from systems.accounts.accounts_stage import Account
accounts=Account()
my_system=System([ fcs, empty_rules, combiner, position_size, raw_data, portfolio, accounts], data, my_config)
profits=my_system.accounts.portfolio()
profits.percent.stats()

2025-03-31 19:34:44 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:34:44 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:34:44 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:34:44 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:34:44 INFO base_system {'stage': 'accounts'} Calculating pandl for portfolio
2025-03-31 19:34:44 DEBUG base_system {'stage': 'positionSize'} Getting vol target
2025-03-31 19:34:44 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'CORN'} Calculating pandl for instrument for CORN
2025-03-31 19:34:44 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating notional position for CORN
2025-03-31 19:34:44 INFO base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating instrument

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:45 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily volatility for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily prices for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating combined forecast for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Calculating forecast weights for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily prices for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily prices for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SOFR
2025-03-31 19:34:45 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOF

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily volatility for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily prices for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Calculating combined forecast for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Calculating forecast weights for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily prices for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500_micro'} Calculating daily prices for SP500_micro
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500_micro'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for S

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily volatility for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating combined forecast for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating forecast weights for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for US10
2025-03-31 19:34:46 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US1

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:34:46 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SOFR'} Calculating pandl for instrument for SOFR
2025-03-31 19:34:46 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating notional position for SOFR
2025-03-31 19:34:46 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} No risk overlay in config: won't apply risk scaling
2025-03-31 19:34:46 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating buffers for SOFR
2025-03-31 19:34:46 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating forecast method buffers for SOFR
2025-03-31 19:34:46 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SOFR'} Calculating buffered positions
2025-03-31 19:34:47 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'SOFR'} Calculating pandl for instrument for SOFR
2025-03-31 19:34:47 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SOFR'} Calculating daily pri

[[('min', '-6.599'),
  ('max', '5.065'),
  ('median', '0.007462'),
  ('mean', '0.01657'),
  ('std', '0.5303'),
  ('skew', '-0.3771'),
  ('ann_mean', '4.242'),
  ('ann_std', '8.485'),
  ('sharpe', '0.4999'),
  ('sortino', '0.6364'),
  ('avg_drawdown', '-7.043'),
  ('time_in_drawdown', '0.9672'),
  ('calmar', '0.2133'),
  ('avg_return_to_drawdown', '0.6023'),
  ('avg_loss', '-0.3551'),
  ('avg_gain', '0.361'),
  ('gaintolossratio', '1.017'),
  ('profitfactor', '1.099'),
  ('hitrate', '0.5195'),
  ('t_stat', '3.62'),
  ('p_value', '0.0002957')],
 ('You can also plot / print:',
  ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]

In [30]:
profits.gross.percent.stats() ## all other things work eg profits.gross.sharpe()
profits.costs.percent.stats()

/Users/henrik/miniconda3/envs/invest/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/henrik/miniconda3/envs/invest/lib/python3.13/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


[[('min', '-0.02195'),
  ('max', '0'),
  ('median', '0'),
  ('mean', '-0.0006931'),
  ('std', '0.001422'),
  ('skew', '-4.645'),
  ('ann_mean', '-0.1774'),
  ('ann_std', '0.02274'),
  ('sharpe', '-7.801'),
  ('sortino', '-6.309'),
  ('avg_drawdown', '-4.724'),
  ('time_in_drawdown', '0.9999'),
  ('calmar', '-0.0191'),
  ('avg_return_to_drawdown', '-0.03756'),
  ('avg_loss', '-0.001397'),
  ('avg_gain', 'nan'),
  ('gaintolossratio', 'nan'),
  ('profitfactor', '0'),
  ('hitrate', '0'),
  ('t_stat', '-56.45'),
  ('p_value', '0')],
 ('You can also plot / print:',
  ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]

In [31]:
from sysdata.config.configdata import Config
my_config=Config(dict(trading_rules=dict(ewmac8=ewmac_8, ewmac32=ewmac_32), instrument_weights=dict(US10=.1, SOFR=.4, CORN=.3, SP500_micro=.2), instrument_div_multiplier=1.5, forecast_scalars=dict(ewmac8=5.3, ewmac32=2.65), forecast_weights=dict(ewmac8=0.5, ewmac32=0.5), forecast_div_multiplier=1.1
,percentage_vol_target=25, notional_trading_capital=500000, base_currency="GBP"))
my_config

Config with elements: base_currency, forecast_div_multiplier, forecast_scalars, forecast_weights, instrument_div_multiplier, instrument_weights, notional_trading_capital, percentage_vol_target, trading_rules

In [33]:
my_config=Config("systems.provided.example.simplesystemconfig.yaml")
my_config

Config with elements: base_currency, forecast_div_multiplier, forecast_weights, instrument_div_multiplier, instrument_weights, notional_trading_capital, percentage_vol_target, trading_rules

In [35]:
from systems.provided.example.simplesystem import simplesystem
my_system=simplesystem()
my_system

my_system.portfolio.get_notional_position("SOFR").tail(5)

2025-03-31 19:37:22 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:37:22 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:37:22 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:37:22 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:37:22 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating notional position for SOFR
2025-03-31 19:37:22 INFO base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating instrument weights
2025-03-31 19:37:22 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Calculating raw instrument weights
2025-03-31 19:37:22 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:37:22 DEBUG base_system Following instrum

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SOFR
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Non-nested dict of forecast weights for SOFR {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Using fixed FDM multiplier of 1.100 for SOFR
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} No mapping applied for SOFR
2025-03-31 19:37:24 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating subsystem position for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating volatility scalar for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating 

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:37:24 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500'} Calculating daily prices for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Non-nested dict of forecast weights for SP500 {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Using fixed FDM multiplier of 1.100 for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} No mapping applied for SP500
2025-03-31 19:37:24 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating subsystem position for US10
2025-03-31 19:37:24 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating volat

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:37:24 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for US10
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Non-nested dict of forecast weights for US10 {'ewmac8': 0.5, 'ewmac32': 0.5}: weights the same for all instruments
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Using fixed FDM multiplier of 1.100 for US10
2025-03-31 19:37:24 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} No mapping applied for US10
2025-03-31 19:37:24 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'SOFR'} Following instruments will have zero weight in optimisation of instrument weights as they have no positions (possibly too expensive?) []
2025-03-31 19:37:24 INFO

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


index
2024-03-22   -4.604817
2024-03-25   -4.425519
2024-03-26   -4.147973
2024-03-27   -3.336962
2024-03-28   -2.914016
Freq: B, dtype: float64

In [36]:
from sysdata.config.configdata import Config
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

my_config=Config("systems.provided.example.simplesystemconfig.yaml")
my_data=csvFuturesSimData()

## I could change my_config, and my_data here if I wanted to
my_system=simplesystem(config=my_config, data=my_data)

2025-03-31 19:37:40 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults


In [37]:
from systems.provided.futures_chapter15.basesystem import futures_system
system=futures_system()
system.portfolio.get_notional_position("EUROSTX").tail(5)

2025-03-31 19:38:05 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:38:05 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:38:05 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:38:05 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:38:05 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Calculating notional position for EUROSTX
2025-03-31 19:38:05 INFO base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Calculating instrument weights
2025-03-31 19:38:05 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Calculating raw instrument weights
2025-03-31 19:38:05 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:38:05 DEBUG base_system Follo

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month
/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'MXP'} Calculating combined forecast for MXP
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'MXP'} Calculating forecast weights for MXP
2025-03-31 19:38:08 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'MXP'} Calculating daily prices for MXP
2025-03-31 19:38:08 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'MXP'} Calculating daily prices for MXP
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'MXP'} Only this set of rules ['carry', 'ewmac16_64', 'ewmac32_128', 'ewmac64_256'] is cheap enough to trade for MXP
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'MXP'} Non-nested dict of forecast weights for MXP {'ewmac16_64': 0.21, 'ewmac32_128': 0.08, 'ewmac64_256': 0.21, 'carry': 0.5}: weights the same for all instruments
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecas

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month
/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} Using fixed FDM multiplier of 1.310 for SOFR
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SOFR'} No mapping applied for SOFR
2025-03-31 19:38:08 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating subsystem position for US10
2025-03-31 19:38:08 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating volatility scalar for US10
2025-03-31 19:38:08 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating instrument value vol for US10
2025-03-31 19:38:08 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'US10'} Calculating instrument currency vol for US10
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating combined forecast for US10
2025-03-31 19:38:08 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'U

/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month
/Users/henrik/dev/pysystemtrade/syscore/pandas/frequency.py:225: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly_index = list(df.resample("1M").last().index)  ## last day in month


2025-03-31 19:38:09 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'V2X'} Using fixed FDM multiplier of 1.310 for V2X
2025-03-31 19:38:09 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'V2X'} No mapping applied for V2X
2025-03-31 19:38:09 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Following instruments will have zero weight in optimisation of instrument weights as they have no positions (possibly too expensive?) []
2025-03-31 19:38:09 INFO base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Using fixed diversification multiplier 1.890000
2025-03-31 19:38:09 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} No risk overlay in config: won't apply risk scaling


index
2024-03-22    5.122159
2024-03-25    5.285780
2024-03-26    5.385521
2024-03-27    5.451916
2024-03-28    5.518672
Freq: B, dtype: float64

In [38]:
from systems.provided.futures_chapter15.estimatedsystem import futures_system
system = futures_system()
system.portfolio.get_notional_position("EUROSTX").tail(5)

2025-03-31 19:38:24 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:38:24 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:38:24 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:38:24 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-03-31 19:38:24 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Calculating notional position for EUROSTX
2025-03-31 19:38:24 INFO base_system {'stage': 'portfolio', 'instrument_code': 'EUROSTX'} Calculating instrument weights
2025-03-31 19:38:24 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-03-31 19:38:24 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-03-31 19:38:24 DEBUG base_system Following

index
2024-03-22    2.133024
2024-03-25    2.183060
2024-03-26    2.207961
2024-03-27    2.231857
2024-03-28    2.264847
Freq: B, dtype: float64

In [39]:
system.cache.pickle("private.this_system_name.pck") ## use any file extension you like

## In a new session
from systems.provided.futures_chapter15.estimatedsystem import futures_system
system = futures_system()
system.cache.unpickle("private.this_system_name.pck")
system.accounts.portfolio().sharpe() 

2025-03-31 19:38:55 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-03-31 19:38:55 INFO base_system {'stage': 'accounts'} Calculating pandl for portfolio
2025-03-31 19:38:55 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'CORN'} Calculating pandl for instrument for CORN
2025-03-31 19:38:55 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating notional position for CORN
2025-03-31 19:38:55 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} No risk overlay in config: won't apply risk scaling
2025-03-31 19:38:55 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating buffers for CORN
2025-03-31 19:38:55 DEBUG base_system {'stage': 'portfolio', 'instrument_code': 'CORN'} Calculating forecast method buffers for CORN
2025-03-31 19:38:55 DEBUG base_system {'stage': 'accounts', 'instrument_code': 'CORN'} Calculating buffered positions
2025-03-31 19:38:55 DEBUG base_system {'stage': 'ac

np.float64(0.5314274131957918)

In [ ]:
profits=system.accounts.portfolio()
profits.percent.stats()
profits.gross.percent.stats()



[[('min', '-21.14'),
  ('max', '9.6'),
  ('median', '0.0128'),
  ('mean', '0.04555'),
  ('std', '1.299'),
  ('skew', '-0.4691'),
  ('ann_mean', '11.66'),
  ('ann_std', '20.79'),
  ('sharpe', '0.561'),
  ('sortino', '0.7272'),
  ('avg_drawdown', '-19.42'),
  ('time_in_drawdown', '0.9651'),
  ('calmar', '0.1754'),
  ('avg_return_to_drawdown', '0.6004'),
  ('avg_loss', '-0.926'),
  ('avg_gain', '0.9171'),
  ('gaintolossratio', '0.9904'),
  ('profitfactor', '1.109'),
  ('hitrate', '0.5283'),
  ('t_stat', '4.062'),
  ('p_value', '4.891e-05')],
 ('You can also plot / print:',
  ['rolling_ann_std', 'drawdown', 'curve', 'percent'])]